In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

chunks_df = pd.read_csv(PROCESSED_DIR / "chunks.csv")
embeddings = np.load(PROCESSED_DIR / "embeddings.npy")

print("Chunks:", len(chunks_df))
print("Embeddings:", embeddings.shape)

Chunks: 27
Embeddings: (27, 384)


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

In [3]:
query = "My payment failed. What should I do?"

query_embedding = model.encode([query])

print("Query embedding shape:", query_embedding.shape)

Query embedding shape: (1, 384)


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    query_embedding,
    embeddings
)[0]

print("Similarity scores:", similarities.shape)

Similarity scores: (27,)


In [5]:
top_indices = similarities.argsort()[::-1][:5]

for rank, index in enumerate(top_indices, start=1):
    print(f"\n{'=' * 70}")
    print(f"Rank: {rank}")
    print(f"Source: {chunks_df.iloc[index]['source']}")
    print(f"Chunk ID: {chunks_df.iloc[index]['chunk_id']}")
    print(f"Similarity: {similarities[index]:.4f}")
    print("\nText:")
    print(chunks_df.iloc[index]["text"])


Rank: 1
Source: payments.md
Chunk ID: 0
Similarity: 0.6506

Text:
# Payment Support

## Payment Failed

If a payment fails, customers should verify that their payment method has sufficient funds and that the billing information is correct.

Customers can try the payment again after checking their payment details. If the payment continues to fail, they should contact customer support.

## Payment Declined

A payment may be declined by the payment provider because of insufficient funds, incorrect payment information, or security restrictions.

Customers should v

Rank: 2
Source: payments.md
Chunk ID: 2
Similarity: 0.5673

Text:
g.

If the payment remains pending for an extended period, they should contact customer support with the transaction details.

## Payment Reversed

A payment may be reversed if the transaction could not be completed successfully. Customers should check their payment history to confirm the current status.

If the amount has not been returned after the expected pro

In [6]:
def retrieve(query, top_k=5):
    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    top_indices = similarities.argsort()[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "source": chunks_df.iloc[index]["source"],
            "chunk_id": int(chunks_df.iloc[index]["chunk_id"]),
            "similarity": float(similarities[index]),
            "text": chunks_df.iloc[index]["text"]
        })

    return results

In [7]:
results = retrieve(
    "My payment failed. What should I do?",
    top_k=3
)

for result in results:
    print("=" * 70)
    print("Source:", result["source"])
    print("Similarity:", round(result["similarity"], 4))
    print(result["text"])

Source: payments.md
Similarity: 0.6506
# Payment Support

## Payment Failed

If a payment fails, customers should verify that their payment method has sufficient funds and that the billing information is correct.

Customers can try the payment again after checking their payment details. If the payment continues to fail, they should contact customer support.

## Payment Declined

A payment may be declined by the payment provider because of insufficient funds, incorrect payment information, or security restrictions.

Customers should v
Source: payments.md
Similarity: 0.5673
g.

If the payment remains pending for an extended period, they should contact customer support with the transaction details.

## Payment Reversed

A payment may be reversed if the transaction could not be completed successfully. Customers should check their payment history to confirm the current status.

If the amount has not been returned after the expected processing period, customers should contact support.

## Pa

In [8]:
test_queries = [
    "I forgot my password. How can I reset it?",
    "I want to return a product.",
    "How long does a refund take?",
    "My internet connection is not working.",
    "Why was I charged twice?"
]

for query in test_queries:
    results = retrieve(query, top_k=1)

    result = results[0]

    print("=" * 70)
    print("QUESTION:", query)
    print("SOURCE:", result["source"])
    print("SIMILARITY:", round(result["similarity"], 4))
    print("TEXT:", result["text"][:300])

QUESTION: I forgot my password. How can I reset it?
SOURCE: account.md
SIMILARITY: 0.533
TEXT: pam or junk folder if the message is not visible in the inbox.

## Forgot Password

Customers who forget their password should use the password reset option on the sign-in page. A password reset link will be sent to the registered email address.

## Password Reset Email Not Received

If the password
QUESTION: I want to return a product.
SOURCE: returns.md
SIMILARITY: 0.6694
TEXT: # Returns Support

## Return Eligibility

Customers may be able to return eligible products according to the applicable return policy.

Return eligibility may depend on the product, purchase date, and condition of the item.

## Start a Return

Customers who want to return a product should contact cu
QUESTION: How long does a refund take?
SOURCE: refunds.md
SIMILARITY: 0.5124
TEXT: t and determine whether the transaction qualifies for a refund.

## Refund Processing

After a refund is approved, the processing time may

In [9]:
retrieval_results = []

for query in test_queries:
    result = retrieve(query, top_k=1)[0]

    retrieval_results.append({
        "query": query,
        "source": result["source"],
        "similarity": result["similarity"]
    })

retrieval_df = pd.DataFrame(retrieval_results)

retrieval_df

,query,source,similarity
0,I forgot my password. How can I reset it?,account.md,0.533033
1,I want to return a product.,returns.md,0.669366
2,How long does a refund take?,refunds.md,0.512385
3,My internet connection is not working.,technical_support.md,0.496682
4,Why was I charged twice?,billing.md,0.523341


In [10]:
retrieval_path = PROCESSED_DIR / "retrieval_test_results.csv"

retrieval_df.to_csv(retrieval_path, index=False)

print("Saved:", retrieval_path)

Saved: d:\ai-customer-support-assistant\data\processed\retrieval_test_results.csv


In [11]:
retrieval_df

,query,source,similarity
0,I forgot my password. How can I reset it?,account.md,0.533033
1,I want to return a product.,returns.md,0.669366
2,How long does a refund take?,refunds.md,0.512385
3,My internet connection is not working.,technical_support.md,0.496682
4,Why was I charged twice?,billing.md,0.523341


In [12]:
print("Average similarity:", round(retrieval_df["similarity"].mean(), 4))
print("Minimum similarity:", round(retrieval_df["similarity"].min(), 4))
print("Maximum similarity:", round(retrieval_df["similarity"].max(), 4))

Average similarity: 0.547
Minimum similarity: 0.4967
Maximum similarity: 0.6694
